Connect To Drive

In [1]:
from google.colab import drive; drive.mount('/content/drive')
import os
import pandas as pd
PROJECT_ROOT = '/content/drive/MyDrive/Projects/multi-agent-discovery'
DATA        = os.path.join(PROJECT_ROOT, 'data/raw/ml-32m')
CATALOG     = os.path.join(PROJECT_ROOT, 'data/processed/catalog/catalog.parquet')
MOVIES_CSV  = os.path.join(DATA, 'movies.csv')

Mounted at /content/drive


Building Tool C

In [2]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_c_details.py
"""Tool C — movie metadata lookup (offline, from the catalog)."""
import pandas as pd


class ToolC:
    def __init__(self, catalog_path, movies_csv):
        self.cat = pd.read_parquet(catalog_path).set_index("movieId")
        self.ml_genres = pd.read_csv(movies_csv).set_index("movieId")["genres"]  # MovieLens genres (complete)

    def _genres(self, movie_id):
        g = self.ml_genres.get(movie_id)
        return [] if (not isinstance(g, str) or g == "(no genres listed)") else g.split("|")

    def get_movie_details(self, movie_id):
        movie_id = int(movie_id)
        if movie_id not in self.cat.index:
            return {"movieId": movie_id, "found": False}
        r = self.cat.loc[movie_id]
        return {
            "movieId": movie_id, "found": True,
            "title":    r["title"],
            "year":     int(r["year"])    if pd.notna(r["year"])    else None,
            "runtime":  int(r["runtime"]) if pd.notna(r["runtime"]) else None,
            "genres":   self._genres(movie_id),
            "director": r["director"],
            "cast":     list(r["cast"]) if r["cast"] is not None else [],
            "overview": r["overview"],
        }

    def get_many(self, movie_ids):
        return [self.get_movie_details(m) for m in movie_ids]

Writing /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_c_details.py


Test Tool C

In [3]:
import sys; sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
sys.modules.pop('tools.tool_c_details', None)
from tools.tool_c_details import ToolC
tool_c = ToolC(CATALOG, MOVIES_CSV)
print(tool_c.get_movie_details(260))     # Star Wars — full record
print(tool_c.get_movie_details(999999))  # nonexistent -> {'found': False}

{'movieId': 260, 'found': True, 'title': 'Star Wars', 'year': 1977, 'runtime': 121, 'genres': ['Action', 'Adventure', 'Sci-Fi'], 'director': 'George Lucas', 'cast': ['Mark Hamill', 'Harrison Ford', 'Carrie Fisher', 'Peter Cushing', 'Alec Guinness'], 'overview': 'Princess Leia is captured and held hostage by the evil Imperial forces in their effort to take over the galactic Empire. Venturesome Luke Skywalker and dashing captain Han Solo team together with the loveable robot duo R2-D2 and C-3PO to rescue the beautiful princess and restore peace and justice in the Empire.'}
{'movieId': 999999, 'found': False}


Building Tool D

In [4]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_d_filter.py
"""Tool D — constrained filter over a candidate movieId list."""
import pandas as pd


class ToolD:
    def __init__(self, catalog_path, movies_csv):
        cat = pd.read_parquet(catalog_path).set_index("movieId")
        self.runtime = cat["runtime"].to_dict()
        self.year    = cat["year"].to_dict()
        ml = pd.read_csv(movies_csv).set_index("movieId")["genres"]
        self.genres = {int(m): set(g.split("|")) for m, g in ml.items()
                       if isinstance(g, str) and g != "(no genres listed)"}

    def filter_by(self, movie_ids, runtime_max=None, runtime_min=None,
                  genre_in=None, year_range=None, exclude_ids=None):
        exclude  = {int(x) for x in (exclude_ids or [])}
        genre_in = set(genre_in) if genre_in else None
        ylo, yhi = year_range or (None, None)
        out = []
        for m in movie_ids:
            m = int(m)
            if m in exclude:
                continue
            rt = self.runtime.get(m); yr = self.year.get(m)
            if runtime_max is not None and (rt is None or pd.isna(rt) or rt > runtime_max): continue
            if runtime_min is not None and (rt is None or pd.isna(rt) or rt < runtime_min): continue
            if year_range is not None:
                if yr is None or pd.isna(yr): continue
                if ylo is not None and yr < ylo: continue
                if yhi is not None and yr > yhi: continue
            if genre_in is not None and not (self.genres.get(m, set()) & genre_in): continue
            out.append(m)
        return {"filtered": out, "n_in": len(movie_ids), "n_out": len(out)}

Writing /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_d_filter.py


Test Tool D

In [5]:
sys.modules.pop('tools.tool_d_filter', None)
from tools.tool_d_filter import ToolD
tool_d = ToolD(CATALOG, MOVIES_CSV)

candidates = [1, 260, 296, 79132, 6377, 364, 2571]   # mixed bag
print(tool_d.filter_by(candidates, runtime_max=120))                      # short films only
print(tool_d.filter_by(candidates, genre_in=["Animation"]))               # animated only
print(tool_d.filter_by(candidates, year_range=(1990, 1999), exclude_ids=[260]))  # 90s, minus SW

{'filtered': [1, 6377, 364], 'n_in': 7, 'n_out': 3}
{'filtered': [1, 6377, 364], 'n_in': 7, 'n_out': 3}
{'filtered': [1, 296, 364, 2571], 'n_in': 7, 'n_out': 4}
